# Phase 7 — 검색·답변 품질, 속도, 사용량 평가

기존 Phase 5 원문·인덱스와 Phase 6 대화 경로를 그대로 평가합니다. 설치나 재색인은 필요하지 않습니다.
기존 가상환경 커널을 선택하고 업데이트 후 Restart하세요.

기본 Run All은 **API 0회**로 개발용 60문항의 검색을 측정합니다.
실제 답변 평가는 아래 `RUN_LIVE_EVAL=True`에서 선택한 소수 사례만 실행합니다.
검색 근거 확보율·인용 유효성·상태 일치는 의미적 정답률과 구분합니다.
기존 holdout 질문 파일은 열지 않습니다. 개발 평가 결과를 새로운 독립 시험 성적으로 표시하지 않습니다.


In [ ]:
from pathlib import Path
import json,sys
from collections import Counter
from IPython.display import display,Markdown

PROJECT_ROOT=None
candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'rfp_phase5_local_20261111']
ROOT=(Path(PROJECT_ROOT).expanduser().resolve() if PROJECT_ROOT else
      next((p.resolve() for p in candidates if (p/'rfp_phase7/evaluate.py').is_file()),None))
if ROOT is None:raise FileNotFoundError('PROJECT_ROOT에 Phase5 프로젝트 폴더를 지정하세요.')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Python:',sys.executable)
print('프로젝트:',ROOT)

from rfp_phase5.runtime import bootstrap_phase5
from rfp_phase7.evidence import load_dev_cases
from rfp_phase7.evaluate import run_evaluation,save_report
from rfp_phase7.review import assess_review
if 'app' in globals() and isinstance(app,dict) and app.get('index') is not None:app['index'].close()
app=bootstrap_phase5(ROOT)
settings=json.loads((ROOT/'configs/phase7.json').read_text(encoding='utf-8'))
cases=load_dev_cases(ROOT/settings['development_questions'])
print('개발 문항:',len(cases),'/ 종류:',dict(Counter(c['feature'] for c in cases)))
print('원본 문서:',len(app['documents']),'/ 대표 문서:',len(app['index'].get_catalog()))


## 1. 평가 범위와 저장된 전체 요약

`OFFLINE_CASE_IDS=None`은 전체 dev 사례입니다. 일부만 확인하려면 사례 ID 목록을 넣습니다.
`SUMMARY_RUN`은 Phase 5의 `content_documents/`가 들어 있는 실행 폴더입니다.
저장된 보정 결과의 지문과 현재 원문·인용을 다시 확인하며, 기존 결과나 검토표를 수정하지 않습니다.
요약 파일이 없는 사례와 합성 파싱 장애 사례는 사유를 붙여 미실행으로 기록합니다.
요약은 상위 검색 청크로 대체하지 않습니다.


In [ ]:
OFFLINE_CASE_IDS=None
SUMMARY_RUN=ROOT/'phase5_outputs/final_source_units/phase5_final_source_units'
if not (SUMMARY_RUN/'content_documents').is_dir():
    print('저장된 content_documents 폴더를 찾지 못했습니다. SUMMARY_RUN을 실제 결과 위치로 바꾸면 요약도 읽습니다.')
display([{'사례':c['case_id'],'종류':c['feature'],
          '질문 턴':len(c.get('turns') or [c['question']])} for c in cases])


## 2. API 없이 검색 비교

- `literal_bm25`: 같은 문서 선택·대화 라우터에서 현재 원래 질문을 BM25로 검색하는 비교 실험입니다.
- `phase6_rrf`: 실제 Phase 6의 맥락 해소·본문 검색어·후보 우선순위·문서별 입력 할당을 사용합니다.

과거 Phase 3 성적과 동일한 실험이라고 간주하지 않습니다.
평가 정답·기대 문서 ID는 검색이 끝난 뒤 채점에만 쓰며, 검색에는 사용자 질문과 `context.selected_doc_ids`만 전달합니다.
근거 확보율은 실제 모델 입력에 들어갈 청크 집합이 필요한 원문 인용의 공백 외 문자 전체를 덮는 비율입니다.
MRR은 첫 관련 원문 겹침의 순위이며, 문장 전체 확보 성공과 구분합니다.
속도는 현재 컴퓨터·이미 열린 인덱스에서 한 번 측정한 값입니다.


In [ ]:
offline_report=run_evaluation(app,cases,profiles=settings['offline_profiles'],
    case_ids=OFFLINE_CASE_IDS,summary_folder=SUMMARY_RUN,
    progress=lambda profile,cid:print(profile,cid,flush=True))
offline_folder=save_report(ROOT,offline_report)
display(offline_report['coverage'])
display(offline_report['alignment'])
display([{'프로필':p,'평가 턴':m['turn_count'],
          '원문근거 확보율':m['source_recovery']['micro_source_span_recall'],
          '상위5개 확보율':m['recall_at_k']['5']['micro_source_span_recall'],
          '미정렬 근거':m['source_recovery']['unmapped_links'],
          'p50(ms)':m['end_to_end_latency']['p50_ms'],
          'p95(ms)':m['end_to_end_latency']['p95_ms'],
          '범위 일치':f"{m['scope_matches']}/{m['scope_evaluable_turns']}"}
         for p,m in offline_report['metrics'].items()])
assert offline_report['usage']['api_calls']==0
print('평가 기록:',offline_folder)
print('의미적 정답률:',offline_report['semantic_accuracy'])


## 3. 누락과 범위 오류 확인

낮은 근거 확보율은 검색 개선 후보입니다. 원문 정렬 실패는 평가 라벨 문제와 구별합니다.
아래 사례 ID로 `report.json`에서 실제 본문 검색어·입력 청크·원문 근거를 확인할 수 있습니다.
숫자 문자열이 보이는 것만으로 정답 처리하지 않습니다.


In [ ]:
diagnostics=[]
for row in offline_report['rows']:
    if row.get('skipped_reason'):continue
    score=row['retrieval_score']['packed_context']
    recall=score['source_span_recall']
    if score['unmapped_links'] or (recall is not None and recall<1) or row.get('scope_match') is False:
        diagnostics.append({'행':row['row_id'],'종류':row['feature'],'근거 확보율':recall,
            '미정렬 근거':score['unmapped_links'],'범위 일치':row['scope_match']})
display(diagnostics[:25])
display([{'행':r['row_id'],'사유':r['skipped_reason']}
         for r in offline_report['rows'] if r.get('skipped_reason')])
display([{'행':r['row_id'],'항목별 상태':r['summary_field_states'],
          '출처 검사':r['citation_audit']}
         for r in offline_report['rows'] if r.get('summary_field_states')])


## 4. 실제 답변 평가 — 선택 실행

기본 사례는 사실 확인 1건, 3턴 후속 질문 1건, 비교 1건, 근거 부족 1건입니다.
최대 6개 질문을 한 생성 클라이언트에서 처리하며 호출 상한은 8회입니다. 비교는 문서별 생성·검증 후 표로 합치므로 한 질문이 여러 호출을 사용할 수 있습니다. 요청 결과의 근거가 없으면 모델 호출 없이 유보할 수 있습니다.
선택·계획을 확인하고 `RUN_LIVE_EVAL=True`로 바꾼 뒤 이 셀을 실행합니다.
`.env`를 기존 방식으로 읽습니다. API 오류가 나면 자동 재시도 없이 멈추고 이전 결과와 사용량을 저장합니다.
새 호출과 로컬 캐시의 응답 시간을 나누어 표시합니다. 저장 요약의 과거 사용량은 이번 비용에 더하지 않습니다.

`PRICING`이 `None`이면 금액은 미산정이고 실제 토큰 사용량을 보고합니다.
공식 단가를 확인한 경우 `model`, `input_per_million`, `cached_input_per_million`,
`output_per_million`, `as_of`, `source`를 지정하면 설정 단가로 USD 추정치를 계산합니다.
추정치는 청구서나 결제 상한이 아닙니다.


In [ ]:
RUN_LIVE_EVAL=False
LIVE_CASE_IDS=settings['live_case_ids']
live_cases=[c for c in cases if c['case_id'] in LIVE_CASE_IDS]
LIVE_APPROVED_DOC_IDS=list(dict.fromkeys(d for c in live_cases for d in c.get('context',{}).get('selected_doc_ids',[])))
PRICING=settings.get('pricing')
DOTENV_PATH=None
live_report=None
live_folder=None

display({'사례':LIVE_CASE_IDS,'문서 허용 목록':LIVE_APPROVED_DOC_IDS,
    '최대 질문 턴':sum(len(c.get('turns') or [c['question']]) for c in live_cases),
    '호출 상한':settings['live_max_calls'],'단가 설정':PRICING})
display([{'문서ID':d['doc_id'],'파일명':d['file_name'],'SHA':d['sha256']}
         for d in app['documents'] if d['doc_id'] in LIVE_APPROVED_DOC_IDS])
if type(RUN_LIVE_EVAL)is not bool:raise TypeError('RUN_LIVE_EVAL은 bool이어야 합니다.')
if RUN_LIVE_EVAL:
    from rfp_phase5_recovery.runner import load_api_key
    load_api_key(ROOT,DOTENV_PATH)
    live_report=run_evaluation(app,cases,profiles=['phase6_rrf'],enabled=True,
        case_ids=LIVE_CASE_IDS,approved_doc_ids=LIVE_APPROVED_DOC_IDS,
        max_calls=settings['live_max_calls'],max_input_bytes=settings['live_max_total_input_bytes'],pricing=PRICING,
        progress=lambda profile,cid:print(profile,cid,flush=True))
    live_folder=save_report(ROOT,live_report)
    for row in live_report['rows']:
        if 'result' not in row:continue
        print(row['row_id'],row['result']['status'],row['result'].get('execution_status'))
        result=row['result']
        if result.get('execution_status') in {'validation_failed','error'}:
            display({'실패 행':row['row_id'],
                '검증 사유':(result.get('validation') or {}).get('errors',[]),
                '문서별 검사':result.get('document_validation',[]),
                '오류 유형':result.get('error_type')})
        display(Markdown(result.get('markdown','')))
    display(live_report['metrics'])
    display(live_report['usage'])
    display(live_report['cost_estimate'])
    print('실제 답변 평가 기록:',live_folder)
    print('이 실행을 검토하려면 아래 검토 집계 셀과 마지막 검증 셀을 순서대로 실행하세요. 추가 API 호출은 없습니다.')
else:print('실제 답변 평가: 미실행 / API 호출: 0회')


## 5. 답변 내용 검토

`report.json`의 실제 답변·인용·입력 청크와 원문을 대조하고, 같은 폴더의 `human_review.json`을 편집합니다.
`reviewer`에 검토자를 적고 각 행의 `judgements`에서 `null`을 `true` 또는 `false`로 기록합니다.
정확성, 인용의 주장 지지 여부, 조건 보존, 필수 사실 누락, 확인 요청·답변 유보의 적절성을 확인합니다.
보여 준 정답 정보는 사람 검토용이며 모델에 전달되지 않았습니다. 결과나 정답을 바꾸면 기존 검토를 재사용할 수 없습니다.

검토가 끝난 뒤 아래 셀만 다시 실행하면 API 호출 없이 집계합니다.
부분 검토의 정확도를 전체 문서 정확도로 표시하지 않습니다. 미실행·오류·누락은 별도 기록을 유지합니다.


In [ ]:
REVIEW_RUN=live_folder if live_folder is not None else offline_folder
review_report=json.loads((REVIEW_RUN/'report.json').read_text(encoding='utf-8'))
review_file=REVIEW_RUN/'human_review.json'
human_review=json.loads(review_file.read_text(encoding='utf-8'))
review_result=assess_review(review_report,human_review)
display(review_result)
print('편집할 검토 파일:',review_file)
print('검토 대상:',len(human_review['rows']),'행')


## 6. 평가 코드 회귀 검사와 실행 상태

이 검사는 평가 정답의 모델 입력 혼입, 잘못된 범위, 원문 일부만 인용한 경우의 과대 채점,
실제 호출·대역·캐시 구분, 실패 요청의 사용량, 검토 기록 연결을 점검합니다.
테스트 통과는 실제 답변 전체의 정답 판정과 구분합니다.


In [ ]:
import subprocess
checks=subprocess.run([sys.executable,'-m','unittest','rfp_phase7.test_evaluation',
    'rfp_phase6.test_session','rfp_phase6.test_retrieval','-q'],cwd=ROOT,capture_output=True,text=True)
print(checks.stderr or checks.stdout)
assert checks.returncode==0,'평가 회귀 검사에 실패했습니다.'
display({'검색 평가':offline_report['status'],
    '실제 답변 평가':live_report['status'] if live_report else 'not_run',
    '사람 검토':review_result['status'],
    '자동 의미적 정답 판정':False,'holdout 질문 파일 열람':False})
print('검색·속도·사용량은 측정값이며, 답변 정확도는 검토 기록을 함께 확인합니다.')
